# EchoHeart — Data Exploration & Ablations
Explore EmpatheticDialogues and DailyDialog datasets, analyze emotion distributions, and run quick inference tests.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

## 1. EmpatheticDialogues — Emotion Distribution

In [ ]:
emp = load_dataset('empathetic_dialogues', split='train', trust_remote_code=True)
df_emp = emp.to_pandas()
print(f'Total rows: {len(df_emp)}')
print(f'Unique conversations: {df_emp["conv_id"].nunique()}')
print(f'Emotion categories: {df_emp["context"].nunique()}')
df_emp.head()

In [ ]:
emotion_counts = df_emp['context'].value_counts()
plt.figure(figsize=(14, 5))
emotion_counts.plot(kind='bar')
plt.title('EmpatheticDialogues — Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. DailyDialog — Utterance Length Distribution

In [ ]:
dd = load_dataset('daily_dialog', split='train', trust_remote_code=True)
all_utterances = [utt for dialog in dd['dialog'] for utt in dialog]
lengths = [len(u.split()) for u in all_utterances]
pd.Series(lengths).describe()

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, edgecolor='black')
plt.title('DailyDialog — Utterance Length (words)')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## 3. Quick Inference Test (base weights, no fine-tuning)

In [ ]:
import torch
from model.model import EchoHeart

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EchoHeart()
model.to(device)
model.eval()
print('Model loaded on', device)

In [ ]:
prompts = [
    ['I feel so lonely and nobody understands me.'],
    ['I got rejected from my dream job. I\'m devastated.'],
    ['Today was actually a really good day!'],
]

for ctx in prompts:
    response = model.generate_response(ctx)
    print(f'User:      {ctx[-1]}')
    print(f'EchoHeart: {response}')
    print()